# Session 2 · Tilt to go where you want

By the end of today, your own code keeps a drone within 5 cm in the wind, flies it through five waypoints, and follows a circle.
You will also change the target, the gains and the wind of a flying drone yourself and watch how it moves.

**How it works**: same as session 1. Type the code with me in the `✍️` cells and check the plot right away with `Shift+Enter`. Blanks are `___`.
The three parts you build today (①②③) run 100 times a second inside the drone's chip. From session 3 on they run in the chips of all six drones.
For cells that print a table, **guess the answer before you run it.**

| Stage | Content |
|---|---|
| 0 | Setup + numpy warm-up |
| 1 | Tilt to move sideways |
| 2 | ① Position controller: position error → wanted acceleration |
| 3 | ② Acceleration → thrust and attitude (picture first) |
| 4 | ③ Tilt limit |
| 5 | First flight: hover in the wind |
| 6 | **Live flight: change things while it flies** |
| 7 | ④ Waypoints: the ring course |
| 8 | Circle trajectory and feed-forward (watch together) |
| 9 | Mission: three flight tests |
| At home | Kalman filter ⑤⑥⑦ (optional) |

## 0. Setup

The PID you built in session 1 (`PID`) is already loaded by the background code. Today we use this PID on **all three axes x, y, z at once**.
Give it an array of length 3 as the error instead of a number, and each axis is computed separately.

In [ ]:
from background import *
set_language("en")

warmup()
pid = PID(kp=6.0)
pid.update(np.array([1.0, -0.5, 0.0]), 0.01)       # the three axis errors at once

**numpy warm-up**: the syntax in today's blanks is all below. Run it, then change the numbers and run it once more.

In [ ]:
a = np.array([3.0, 4.0, 0.0])          # a three-slot x, y, z array (positions, velocities, accelerations all look like this)
b = np.array([1.0, 0.0, 2.0])
print("subtract       a - b       =", a - b)
print("scale all      2 * a       =", 2 * a)
print("first two (xy) a[:2]       =", a[:2])
print("length         norm(a)     =", np.linalg.norm(a), "  # √(3² + 4² + 0²)")
print("2-slot length  hypot(3, 4) =", np.hypot(3, 4))
c = a.copy(); c[:2] *= 0.5             # halve only the first two slots (used in stage 4)
print("halve first two            =", c)
st = {"p": a, "v": b}                  # a bundle you read by label (a dictionary)
print('by label       st["p"]     =', st["p"])

## 1. Tilt to move sideways

A drone's thrust always points out of the top of its body. To move sideways it **tilts and turns part of the thrust sideways.**
Just like leaning your body on a kick scooter.

To hold the height, the upward part of the thrust must equal the weight: $T\cos\theta = mg$.
The sideways part is then $T\sin\theta = mg\tan\theta$; divide by the mass for the sideways acceleration

$$a_x = g\tan\theta$$

✍️ Return the sideways acceleration produced by a tilt θ [rad]. (`np.tan`)

In [ ]:
def sideways_acc(theta, g=G):
    return ___  # ✍️


tilt_test(sideways_acc)

**Look and think**
- At first the solid line follows the dashed line (my formula) as the speed builds up. The bigger the tilt, the faster it builds.
- As it speeds up, the solid line sags below the dashed one, because the simulator also computes **air drag**.
  Tilting by the formula alone does not get you where you want. So we need feedback that keeps watching the position and correcting (the same story as session 1).

So the chip is split in two.
- **Outer loop (the manager)**: where to go and how urgently → decides the wanted **acceleration**. Today's ①
- **Inner loop (the driver)**: how much to tilt to get that acceleration → turns it into **thrust and attitude**. Today's ②③
- The attitude controller that actually makes that attitude is already written (500 times a second).

## 2. ① Position controller: position error → wanted acceleration

We add two things to the session-1 PID.

$$\mathbf a = \underbrace{K_p\,\mathbf e_p + K_i\!\int\!\mathbf e_p\,dt}_{\text{session-1 PID (P, I)}} + \underbrace{K_d\,\mathbf e_v}_{\text{brake}} + \underbrace{\mathbf a_{ff}}_{\text{feed-forward}}$$

- $\mathbf e_p$ = target position − current position, $\mathbf e_v$ = target velocity − current velocity
- **D on the velocity error**: the derivative doesn't spike even if the target jumps.
- **Feed-forward** $\mathbf a_{ff}$: when the trajectory says in advance "accelerate this much right now", add it as is (you'll see the effect in stage 8).

`st` is the current state, `sp` is the target (setpoint). Read both like `st["p"]`, `sp["v"]`; the values are arrays of length 3.

**Guess before you run** (`kp = 6`, `kd = 4`, `ki = 0`): if the target is 1 m away toward +x, what is a? If the drone is on target but moving 1 m/s toward +y?

✍️ Fill in the two errors and the output.

In [ ]:
class PositionController:
    def __init__(self, kp=6.0, kd=4.0, ki=2.0, i_limit=0.4, i_band=0.3):
        self.pid = PID(kp, ki, 0.0, i_limit, i_band)     # P and I on e_p (week-1 PID)
        self.kd = kd                                     # D multiplies the velocity error

    def reset(self):
        self.pid.reset()

    def acceleration(self, st, sp, dt):
        e_p = ___  # ✍️
        e_v = ___  # ✍️
        return ___  # ✍️


check_acceleration(PositionController)

Second row: on target but moving toward +y, so it brakes the other way (−y).

## 3. ② Acceleration → thrust and attitude

An elevator floor pushes the holding part ($g$) plus the accelerating part ($a$). A drone is the same. The force it must produce is

$$\mathbf f = m(\mathbf a + g\,\hat z)$$

Turn the top of the body (where the thrust comes out) toward this force. Let's look at a picture before the code.

In [ ]:
force_picture(acc=[5.0, 3.0, 0.0])          # wanted acceleration [m/s²]. Change the numbers and run it again

**Look and think**
- The blue arrow f is the direction the thrust must point. The body (black bar) lies perpendicular to that arrow.
- From the side, the forward lean is **pitch = atan2(f_x, f_z)**: how big the forward part is compared with the upward part.
- From behind, the sideways lean is **roll = atan2(−f_y, √(f_x² + f_z²))**. After leaning forward, the upward part is √(f_x² + f_z²).
  To go left (+y) the drone lowers its left side; by the right-hand rule that is a negative roll, hence −f_y.
- Thrust **T = |f|**: the more it tilts, the harder it must push to keep the upward part.
- Change `acc` to `[9.81, 0, 0]`, `[0, 5, 0]`, `[0, 0, 3]`. How do the angles and the thrust change?

Now into code. If the drone faces another way (yaw), we first rotate by −yaw and read the force in the body's frame (front, left);
**those two lines are already filled in.** With yaw = 0, `fx = f[0]` and `fy = f[1]`, exactly as in the picture above.

**Guess before you run**: if we ask for g of acceleration toward +x, what is the pitch, and how many times the hover thrust is the thrust?
If we ask for 20 m/s² forward (about twice gravity), how far does it tilt? About 30°, 45°, 64°, 90°?

✍️ Fill in four lines: force, pitch, roll, thrust. (`np.arctan2`, `np.hypot`, `np.linalg.norm`)

In [ ]:
def acc_to_thrust_attitude(acc, yaw, mass):
    f = ___  # ✍️ m·(a + g ẑ)
    c, s = np.cos(yaw), np.sin(yaw)
    fx = c * f[0] + s * f[1]             # front part, rotated by −yaw (f[0] when yaw = 0)
    fy = -s * f[0] + c * f[1]            # left part, rotated by −yaw (f[1] when yaw = 0)
    pitch = ___  # ✍️ the angle in the side view
    roll = ___  # ✍️ the angle in the back view (mind the sign)
    thrust = ___  # ✍️ the size of the whole force
    return thrust, np.array([roll, pitch, yaw])


check_attitude(acc_to_thrust_attitude)

Lying 64° over, the upward part of the thrust is too small to hold the height. So the next stage limits the tilt.

## 4. ③ Tilt limit

Just as you don't lean a bicycle past a point, the drone tilts only up to 30°.
Keep the height (a_z) and shrink **only the size of the sideways acceleration**, x and y by the same ratio (stage 0's "halve first two").

- Upward acceleration available: `a_up = a_z + g` (never below 0.1g, even when descending fast)
- Sideways acceleration possible at 30°: `h_max = a_up · tan(30°)`
- If the current sideways acceleration `h = √(a_x² + a_y²)` exceeds `h_max`, multiply `acc[:2]` by `h_max / h`

✍️ Fill in the last two lines.

In [ ]:
def limit_tilt(acc, max_tilt):
    acc = np.array(acc, float)
    a_up = max(acc[2] + G, 0.1 * G)
    h_max = a_up * np.tan(max_tilt)
    h = np.hypot(acc[0], acc[1])
    if ___:  # ✍️
        acc[:2] *= ___  # ✍️
    return acc


check_tilt(limit_tilt, acc_to_thrust_attitude)

## 5. First flight: hover in the wind

With ①②③ all in place, the chip is complete. Put them in the chip and fly.
A wind of about 3% of the weight keeps pushing, with gusts mixed in. Hover at (0, 0, 1); the position error is measured from 5 s on.

The cell below compares the default gains with **the position I term turned off (PD only)**.

In [ ]:
chip = MyChip(PositionController, acc_to_thrust_attitude, limit_tilt)

hover_in_wind(chip, default={}, PD_only={"ki": 0})

**Look and think**
- With PD only, it stays pushed toward the wind. For P to hold the wind's ≈ 0.3 m/s², an error of 0.3 / 6 ≈ 5 cm must remain.
- I removes that remaining error. The same story as session 1's "sag of the heavy drone".

## 6. Live flight: change things while it flies

Keep the drone with your chip flying **at 1× speed** and change values with the sliders while it flies (up to 3 minutes; ■ Stop ends it).
**Guess first** before each change, then check the plots.

| Try | Guess first |
|---|---|
| Jump target x from 0 → 1.0 | How far does it tilt? Does it hit stage 4's 30° limit? |
| Raise the wind to 0.03 N | Does the drone lean into the wind or away from it? |
| Keep the wind and set ki to 0 | Does it come back to the target or stay pushed? (stage 5's PD only) |
| Lower kd to 1 and move target x | Does it stop right at the target, or overshoot and come back? |
| Turn on "fly a circle" | Does it follow the target (dashed) on time? (stage 8 shows why) |

If you lower kd all the way to 0, the wobble grows until it flies off. If you tried it, set it back to 4 at once.

The same values go to PlotJuggler under /pos/... . Open the window with the second cell below (Yes → OK). For a 3D window use `window=True`.

In [ ]:
live(chip)

In [ ]:
plotjuggler("week02")

## 7. ④ Waypoints: the ring course

Now the ground-station (laptop) code. Like a navigator: "You have reached the waypoint, starting the next route."
When the drone comes within a 15 cm radius of the current target, move on to the next point. The ground station calls `update(current position)` 50 times a second
and sends the returned point to the drone.

✍️ Fill in the arrival check. Don't check after finishing (`self.done`), so `self.wps[self.i]` never raises an error.

In [ ]:
class WaypointFollower:
    def __init__(self, waypoints, radius=0.15):
        self.wps = np.asarray(waypoints, float)
        self.radius = radius
        self.i = 0                       # index of the waypoint we head for

    @property
    def done(self):
        return self.i >= len(self.wps)

    def update(self, p):
        if not self.done and ___ < self.radius:  # ✍️ distance to the current target
            ___  # ✍️
        return self.wps[min(self.i, len(self.wps) - 1)]


ring_course(chip, WaypointFollower)

**Look and think**
- The command stops at 30°, yet the maximum tilt goes slightly past 30°: the real attitude overshoots a little while following the command.
- ④ runs on the ground station, ①②③ in the chip. Formation, consensus and avoidance in sessions 3–5 are ground-station code too.
- Weaken the brake like `ring_course(chip, WaypointFollower, {"kd": 1})`: what happens at each point? Is it what you saw in stage 6?

## 8. Circle trajectory and feed-forward (watch together)

In stage 6, with the circle on, the drone followed the target late, because we gave only the target position.

A circle is known in advance: at every point on it we know the velocity (along the tangent) and the acceleration (toward the centre).
Like turning the wheel early when you see the bend, add the acceleration the trajectory tells us as ①'s last term ($\mathbf a_{ff}$).
The circle trajectory is uploaded to the chip and the chip runs it itself (the same as Crazyflie's trajectory upload).

The cell below compares a flight with feed-forward on and one with it off.

In [ ]:
circle(chip)

**Look and think**: without feed-forward the drone is pushed outside the circle and flies around off the path, like a car that turns the wheel only after it has drifted outward.

## 9. Mission: three flight tests

Success means all four boxes green with the default gains (kp 6, kd 4, ki 2).

| Test | Criterion |
|---|---|
| Hover in the wind | position error RMS < 5 cm |
| Ring course | 5/5 finished, within 40 s, max tilt ≤ 35° |
| Circle trajectory | RMS < 10 cm |

In [ ]:
flight_tests(chip, WaypointFollower)

If you finished early, change the gains with the sliders and fly the tests again. The order is the same as session 1: `kp` for speed, `kd` for the wobble, `ki` for the remaining error.
Which gains fly the ring course fastest while keeping all four boxes green?

In [ ]:
tune(chip, WaypointFollower)

## Recap

| What we used today | In one line |
|---|---|
| Tilt | Tilt to move sideways. $a_x = g\tan\theta$ |
| ① Position controller | $\mathbf a = \text{PID}(\mathbf e_p) + K_d\mathbf e_v + \mathbf a_{ff}$ |
| ② Thrust and attitude | Turn the body toward $\mathbf f = m(\mathbf a + g\hat z)$. Thrust is $|\mathbf f|$ |
| ③ Tilt limit | Past 30°, shrink only the sideways acceleration |
| Live flight | Lean into the wind to hold; I removes the remaining error, D calms the wobble |
| ④ Waypoints | Inside the radius → next point |
| Feed-forward | Add the acceleration you know in advance and you won't fall behind |

**One question before you leave**: when the drone holds its place in the wind without being pushed, is the body level or tilted? Why?

Next session the chip you built today runs on **all six drones**, and the ground station tells the six to make a formation.

---
## At home (optional, 25 min): a real chip doesn't know the truth · Kalman filter

We don't do this in class. If you finished stages 1–9, continue at home. You won't need it for session 3.

So far the chip received the **true** position that only the simulator knows. A real chip has just two things.

| Sensor | Rate | Weakness |
|---|---|---|
| IMU (accelerometer) | 500 Hz | bias 0.1 m/s². Integrated, it drifts 5 m in 10 s |
| Ceiling camera | 100 Hz | noise 5 mm, delay 20 ms |

Guessing your position by counting steps with your eyes closed (IMU) drifts more and more; opening your eyes now and then (camera) fixes it. **Mix the two.**
We build a 1-D Kalman filter that estimates only the altitude. The state is `x = [altitude z, vertical speed vz]`, the uncertainty `P` (2×2).

- ⑤ **Predict**: predict dt ahead with the IMU acceleration. `x ← F·x + B·a`, `P ← F·P·Fᵀ + q·B·Bᵀ`
- ⑥ **Kalman gain**: how much to trust the camera. `K = P·Hᵀ / (H·P·Hᵀ + r)` (the camera measures only altitude, so `H = [1, 0]`)
- ⑦ **Correct**: `x ← x + K·(z_cam − H·x)`, `P ← (I − K·H)·P`

✍️ Fill in five lines. Matrix product is `@`, outer product `np.outer`.

In [ ]:
class AltitudeKF:
    def __init__(self, z0=0.0, accel_noise=0.2, cam_noise=0.005):
        self.x = np.array([z0, 0.0])
        self.P = np.diag([1e-3, 1e-2])
        self.q = accel_noise ** 2          # accelerometer noise variance
        self.r = cam_noise ** 2            # camera noise variance

    @property
    def z(self):
        return float(self.x[0])

    def predict(self, a, dt):
        F = np.array([[1.0, dt], [0.0, 1.0]])
        B = np.array([0.5 * dt * dt, dt])
        self.x = ___  # ✍️
        self.P = ___  # ✍️

    def update(self, z_meas):
        H = np.array([1.0, 0.0])
        S = H @ self.P @ H + self.r
        K = ___  # ✍️
        self.x = ___  # ✍️
        self.P = ___  # ✍️


check_kf(AltitudeKF)

Now we only change what measures the altitude, and fly the session-1 altitude controller.

In [ ]:
kf_test(AltitudeKF)

**Look and think**: using the camera value as is, the D term differentiates the 5 mm noise and the thrust jitters every step (orange, right).
The Kalman filter cuts that jitter by a factor of tens, and the altitude error is smaller than with the raw camera too.
On a real drone this jitter comes back as motor noise, heat and battery drain.